# VGG16-BN seed 1 — methods from existing theta_P

Attach CIFAR-100 and the prior output containing the completed VGG16 seed-1
`plateau_checkpoint.pt`. This notebook does **not** train Vanilla. It verifies
the full checkpoint state and SHA-256, then starts both methods from the same
byte-identical theta_P:

- GPU0: recurrent E-driven O, patience 15, 150 SGD epochs;
- GPU1: one O-only projection at theta_P, then 150 uninterrupted SGD epochs.

This is a VGG hyperparameter-sensitivity run: line-search scales
`{0.025, 0.05, 0.1, 0.2, 0.4}` and 64 projection samples. Reporting tracks
strict accuracy best; recurrent search anchors on accuracy then lower loss.
The full anchor checkpoint includes model, optimizer, scheduler, RNG, and
loader state. Both projected arms
preserve the optimizer, scheduler, and LR inherited from theta_P; no LR is
changed after intervention.

Old recurrent O-only checkpoints are rejected. The official CIFAR-100 test set
is never constructed.


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, threading, time, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MAIN_URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
REPO = Path("/kaggle/working/counterfactual-projection")
REFERENCE_URL = "https://github.com/duyh80456-code/One-Shot-TAS-CCIL.git"
REFERENCE = Path("/kaggle/working/One-Shot-TAS-CCIL")
GROMO_URL = "https://github.com/growingnet/gromo.git"
GROMO_COMMIT = "8d19107b61a9459a9021065a329b699adcb0f25b"
GROMO = Path("/kaggle/working/gromo")
RUN_ROOT = Path("/kaggle/working/vgg16_seed1_methods_from_plateau_v3")
OUTPUT = RUN_ROOT / "phase2"

def private_clone(url, destination, branch):
    token = UserSecretsClient().get_secret("github_token").strip()
    if not token: raise RuntimeError("Kaggle Secret github_token is unavailable")
    askpass = Path("/kaggle/working/.counterfactual_git_askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass),
               GIT_TERMINAL_PROMPT="0")
    try:
        subprocess.run(["git", "clone", "--branch", branch, "--single-branch",
                        url, str(destination)], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

for checkout in (REPO, REFERENCE, GROMO):
    if checkout.exists(): shutil.rmtree(checkout)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, "main")
private_clone(REFERENCE_URL, REFERENCE, "ccil-residual-capacity")
subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                GROMO_URL, str(GROMO)], check=True)
subprocess.run(["git", "-C", str(GROMO), "fetch", "--depth", "1",
                "origin", GROMO_COMMIT], check=True)
subprocess.run(["git", "-C", str(GROMO), "checkout", "--detach",
                GROMO_COMMIT], check=True)
print("main:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(GROMO)], check=True)
GROMO_SRC = GROMO / "src"
RUNTIME_PYTHONPATH = os.pathsep.join(filter(None, (
    str(REPO), str(GROMO_SRC), str(REFERENCE), os.environ.get("PYTHONPATH", ""))))
test_env = os.environ.copy(); test_env.update(PYTHONPATH=RUNTIME_PYTHONPATH)
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO,
               env=test_env, check=True)
import torch
if torch.cuda.device_count() != 2:
    raise RuntimeError(f"Select T4 x2; found {torch.cuda.device_count()} GPU(s)")
required_cifar_files = {"train", "test", "meta"}
cifar_dirs = []
for directory, _, filenames in os.walk("/kaggle/input", followlinks=True):
    path = Path(directory)
    if (path.name == "cifar-100-python" and
            required_cifar_files.issubset(filenames)):
        cifar_dirs.append(path.parent)
cifar_dirs = sorted(set(cifar_dirs), key=str)
if not cifar_dirs:
    visible = []
    for directory, dirnames, filenames in os.walk(
            "/kaggle/input", followlinks=True):
        visible.append({
            "directory": directory,
            "directories": sorted(dirnames)[:10],
            "files": sorted(filenames)[:10],
        })
        if len(visible) == 20:
            break
    raise FileNotFoundError(
        "Attach CIFAR-100 containing cifar-100-python/{train,test,meta}. "
        f"Visible input inventory: {visible}")
DATA_ROOT = cifar_dirs[0]
print("CIFAR-100 root:", DATA_ROOT)


In [ ]:
import importlib.util

def load_checkout_module(name, relative_path):
    path = REPO / relative_path
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot import repository helper: {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

discover_checkpoints = load_checkout_module(
    "_counterfactual_kaggle_checkpoint_discovery",
    "experiments/kaggle_checkpoint_discovery.py").discover_checkpoints

SEED = 1
schedule_id = "cifar-vgg16-bn-sgd-multistep-200-v1-post200-val-best"
candidates, rejected = discover_checkpoints(
    "/kaggle/input", RUN_ROOT / "plateau_discovery_cache",
    kind="plateau_fork_checkpoint")
compatible = []
required = {
    "model", "optimizer", "scheduler", "rng",
    "train_loader_generator_state", "train_indices", "trigger_indices",
    "evaluation_indices", "source_tuning_indices", "history", "epoch",
}
for item in candidates:
    payload = item["payload"]
    protocol = payload.get("protocol", {})
    if (required.issubset(payload)
            and protocol.get("seed") == SEED
            and protocol.get("dataset") == "CIFAR-100"
            and protocol.get("architecture") == "CIFAR-VGG16-BN"
            and protocol.get("schedule_id") == schedule_id
            and payload.get("vanilla_baseline_complete") is True
            and payload.get("vanilla_control", {}).get(
                "post_fork_epochs") == 150):
        compatible.append(item)

if not compatible:
    inventory = [{
        "kind": item["payload"].get("kind"),
        "epoch": item["payload"].get("epoch"),
        "seed": item["payload"].get("protocol", {}).get("seed"),
        "architecture": item["payload"].get("protocol", {}).get(
            "architecture"),
        "schedule_id": item["payload"].get("protocol", {}).get(
            "schedule_id"),
    } for item in candidates]
    raise FileNotFoundError(
        "Attach a completed VGG16 seed-1 plateau_checkpoint.pt. "
        f"Visible plateau inventory: {inventory}; rejected: {rejected}")

by_hash = {}
for item in compatible:
    by_hash.setdefault(item["sha256"], item)
if len(by_hash) != 1:
    raise RuntimeError(
        "Multiple distinct compatible VGG16 seed-1 theta_P checkpoints are "
        f"attached: {sorted(by_hash)}. Attach exactly one run.")

selected = next(iter(by_hash.values()))
PLATEAU_CHECKPOINT = Path(selected["path"])
PLATEAU_HASH = selected["sha256"]
PLATEAU_PAYLOAD = selected["payload"]
PLATEAU_EPOCH = int(PLATEAU_PAYLOAD["epoch"])
VANILLA_CONTROL = dict(PLATEAU_PAYLOAD["vanilla_control"])
if hashlib.sha256(PLATEAU_CHECKPOINT.read_bytes()).hexdigest() != PLATEAU_HASH:
    raise RuntimeError("theta_P changed after checkpoint discovery")
print("Loaded VGG16 seed-1 theta_P:", {
    "epoch": PLATEAU_EPOCH,
    "sha256": PLATEAU_HASH,
    "path": str(PLATEAU_CHECKPOINT),
    "vanilla_best": VANILLA_CONTROL.get("best_validation_accuracy"),
    "vanilla_final": VANILLA_CONTROL.get("final_validation_accuracy"),
})


## Run methods directly from the attached theta_P

No Phase-1 command is executed below.


In [ ]:
def base_args(output):
    return [
        "--reference-root", str(REFERENCE), "--data-root", str(DATA_ROOT),
        "--plateau-checkpoint", str(PLATEAU_CHECKPOINT),
        "--plateau-checkpoint-hash", PLATEAU_HASH,
        "--output", str(output), "--post-fork-epochs", "150",
        "--seed", "1", "--architecture", "vgg16",
        "--site", "auto",
        "--o-only-site", "stages.2.links.0",
        "--site-selection-mode", "all_functional_gain",
        "--batch-size", "64", "--rank", "4",
        "--probe-epsilon", "0.05", "--where-batches", "3",
        "--line-search-scales", "0.025,0.05,0.1,0.2,0.4",
        "--projection-samples", "64"]

method_names = ("ours_e_driven_o", "o_projection_only")
retrigger_patience = {
    "ours_e_driven_o": 15,
    "o_projection_only": 10,
}
commands = {
    name: [sys.executable, "-m", "experiments.run_plateau_fork",
           "--method", name] + base_args(OUTPUT / name) + [
               "--retrigger-patience", str(retrigger_patience[name])]
    for name in method_names}

# Resume only arm checkpoints produced from this exact theta_P and protocol.
arm_checkpoints, rejected_arm_checkpoints = discover_checkpoints(
    "/kaggle/input", OUTPUT,
    kind={"plateau_fork_arm_progress", "plateau_fork_arm_best"})
arm_progress = [item for item in arm_checkpoints
                if item["payload"]["kind"] == "plateau_fork_arm_progress"]
arm_bests = [item for item in arm_checkpoints
             if item["payload"]["kind"] == "plateau_fork_arm_best"]
for name in method_names:
    expected_schedule_mode = {
        "ours_e_driven_o": "recurrent_best_rollback",
        "o_projection_only": "single_initial_intervention",
    }[name]
    matches = [item for item in [*arm_progress, *arm_bests]
               if item["payload"].get("theta_best_hash") == PLATEAU_HASH
               and item["payload"].get("protocol", {}).get("method") == name
               and item["payload"].get("protocol", {}).get("architecture") ==
                   "CIFAR-VGG16-BN"
               and (item["payload"].get("protocol", {}).get(
                   "intervention_schedule") or {}).get("mode") ==
                   expected_schedule_mode
               and (name != "ours_e_driven_o" or
                    (item["payload"].get("protocol", {}).get(
                        "intervention_schedule") or {}).get("patience") ==
                    retrigger_patience[name])
               and item["payload"].get("protocol", {}).get(
                   "protocol_version") == 3
               and item["payload"].get("protocol", {}).get(
                   "intervention_config") == {
                       "projection_samples": 64,
                       "line_search_scales": [0.025, 0.05, 0.1, 0.2, 0.4]}]
    if not matches:
        continue
    progress = max(
        matches, key=lambda item: int(item["payload"]["post_fork_epoch"]))
    best_matches = [item for item in arm_bests
                    if item["payload"].get("theta_best_hash") == PLATEAU_HASH
                    and item["payload"].get("protocol") ==
                    progress["payload"].get("protocol")]
    if not best_matches:
        raise RuntimeError(
            f"Found resumable {name} progress without its best checkpoint")
    best = max(
        best_matches,
        key=lambda item: int(item["payload"]["post_fork_epoch"]))
    arm_output = OUTPUT / name
    arm_output.mkdir(parents=True, exist_ok=True)
    shutil.copy2(progress["path"], arm_output / "checkpoint_latest.pt")
    shutil.copy2(best["path"], arm_output / "checkpoint_best.pt")
    print(f"Resuming {name} at post-fork epoch "
          f"{progress['payload']['post_fork_epoch']}")

def stream(name, process, log):
    for line in process.stdout:
        log.write(line); log.flush()
        print(f"[{name}] {line}", end="", flush=True)

def launch(gpu, name):
    out = OUTPUT / name; out.mkdir(parents=True, exist_ok=True)
    log = (out / "run.log").open("a", buffering=1)
    env = os.environ.copy()
    env.update(CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1",
               PYTHONPATH=RUNTIME_PYTHONPATH)
    process = subprocess.Popen(
        commands[name], cwd=REPO, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1)
    thread = threading.Thread(
        target=stream, args=(name, process, log), daemon=True)
    thread.start()
    print(f"GPU{gpu}: started {name}, pid={process.pid}")
    return name, process, thread, log

def finish(job):
    name, process, thread, log = job
    code = process.wait()
    thread.join(); log.close()
    print(name, "exit=", code)
    return name, code

print("GPU0: E-driven O; GPU1: O-only")
jobs = [launch(0, "ours_e_driven_o"),
        launch(1, "o_projection_only")]
statuses = [finish(job) for job in jobs]
failures = [status for status in statuses if status[1]]
if failures:
    raise RuntimeError(f"failed arms: {failures}")


In [ ]:
results = {
    name: json.loads((OUTPUT / name / "result.json").read_text())
    for name in ("ours_e_driven_o", "o_projection_only")}
for name, result in results.items():
    if result["plateau_checkpoint_hash"] != PLATEAU_HASH:
        raise RuntimeError(f"{name} used another theta_P")
    if result["theta_best_hash"] != PLATEAU_HASH:
        raise RuntimeError(f"{name} used another theta_best")
    if result["post_fork_epochs"] != 150:
        raise RuntimeError(f"{name} did not complete 150 epochs")
    if result.get("protocol", {}).get("architecture") != "CIFAR-VGG16-BN":
        raise RuntimeError(f"{name} did not run CIFAR-VGG16-BN")
    if not Path(result["best_checkpoint"]).is_file():
        raise RuntimeError(f"{name} did not save checkpoint_best.pt")

VANILLA_CONTROL["plateau_checkpoint_hash"] = PLATEAU_HASH
VANILLA_CONTROL["theta_best_hash"] = PLATEAU_HASH
VANILLA_CONTROL["best_checkpoint"] = str(PLATEAU_CHECKPOINT)
results["vanilla"] = VANILLA_CONTROL
metric_fields = (
    "fork_validation_accuracy", "fork_validation_loss",
    "theta_P_validation_accuracy", "theta_P_validation_loss",
    "final_validation_accuracy", "best_validation_accuracy",
    "final_validation_loss", "best_validation_loss",
    "validation_accuracy_delta", "best_validation_accuracy_delta",
    "epochs_to_best", "training_seconds", "peak_gpu_memory",
    "peak_train_params", "deploy_params", "best_checkpoint",
    "intervention_count", "correction_application_count",
    "correction_application_rate", "intervention_seconds",
    "rollback_count", "retrigger_patience", "interventions")
summary = {
    "architecture": "CIFAR-VGG16-BN", "seed": SEED,
    "plateau_epoch": PLATEAU_EPOCH,
    "stall_patience": 150,
    "plateau_checkpoint_hash": PLATEAU_HASH,
    "theta_best_hash": PLATEAU_HASH,
    "post_fork_epochs": 150, "official_test_used": False,
    "phase1_vanilla_control": VANILLA_CONTROL,
    "results": {
        name: {key: result.get(key) for key in metric_fields}
        for name, result in results.items()},
}
(OUTPUT / "summary.json").write_text(
    json.dumps(summary, indent=2, sort_keys=True))
print(json.dumps(summary, indent=2, sort_keys=True))
print("\nmethod                  fork_acc   best_acc  final_acc  epoch_best")
def metric(value):
    return "n/a" if value is None else f"{value:.4f}"

for name in ("vanilla", "ours_e_driven_o", "o_projection_only"):
    row = results[name]
    print(f"{name:23s} {metric(row['fork_validation_accuracy']):>8s} "
          f"{metric(row['best_validation_accuracy']):>9s} "
          f"{metric(row['final_validation_accuracy']):>10s} "
          f"{str(row['epochs_to_best']):>10s}")
archive = shutil.make_archive(str(OUTPUT), "gztar", root_dir=OUTPUT)
print("Archive:", archive)


In [ ]:
archive = shutil.make_archive(str(RUN_ROOT), "gztar", root_dir=RUN_ROOT)
print("Method-only archive:", archive)
print("Loaded fork checkpoint:", PLATEAU_CHECKPOINT)
